### О задании

В этом задании вам предстоит предсказывать год выпуска песни (**задача регрессии**) по некоторым звуковым признакам: [данные](https://archive.ics.uci.edu/ml/datasets/yearpredictionmsd). В ячейках ниже находится код для загрузки данных. Обратите внимание, что обучающая и тестовая выборки располагаются в одном файле, поэтому НЕ меняйте ячейку, в которой производится деление данных.

In [1]:
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

In [ ]:
# %conda install wget

In [2]:
!wget -O data.txt.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip

--2025-03-25 13:12:01--  https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘data.txt.zip’

data.txt.zip            [    <=>             ] 201.24M  13.9MB/s    in 25s     

2025-03-25 13:12:27 (7.90 MB/s) - ‘data.txt.zip’ saved [211011981]



In [3]:
df = pd.read_csv("data.txt.zip", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,81,82,83,84,85,86,87,88,89,90
0,2001,49.94357,21.47114,73.07750,8.74861,-17.40628,-13.09905,-25.01202,-12.23257,7.83089,...,13.01620,-54.40548,58.99367,15.37344,1.11144,-23.08793,68.40795,-1.82223,-27.46348,2.26327
1,2001,48.73215,18.42930,70.32679,12.94636,-10.32437,-24.83777,8.76630,-0.92019,18.76548,...,5.66812,-19.68073,33.04964,42.87836,-9.90378,-32.22788,70.49388,12.04941,58.43453,26.92061
2,2001,50.95714,31.85602,55.81851,13.41693,-6.57898,-18.54940,-3.27872,-2.35035,16.07017,...,3.03800,26.05866,-50.92779,10.93792,-0.07568,43.20130,-115.00698,-0.05859,39.67068,-0.66345
3,2001,48.24750,-1.89837,36.29772,2.58776,0.97170,-26.21683,5.05097,-10.34124,3.55005,...,34.57337,-171.70734,-16.96705,-46.67617,-12.51516,82.58061,-72.08993,9.90558,199.62971,18.85382
4,2001,50.97020,42.20998,67.09964,8.46791,-15.85279,-16.81409,-12.48207,-9.37636,12.63699,...,9.92661,-55.95724,64.92712,-17.72522,-1.49237,-7.50035,51.76631,7.88713,55.66926,28.74903


In [4]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

train_size = 463715
X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

## Задание 0. (0 баллов, но при невыполнении максимум за все задание &mdash; 0 баллов)

Мы будем использовать RMSE как метрику качества. Для самого первого бейзлайна обучите `Ridge` регрессию из `sklearn`. Кроме того, посчитайте качество при наилучшем константном прогнозе.

In [5]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [6]:
model = Ridge (alpha=1.0)
model.fit (X_train, y_train)

y_pred = model.predict (X_test)

rmse = mean_squared_error (y_test, y_pred) ** 0.5
print ('rmse: ', rmse)

rmse:  9.510160711373397


## Задание 1. (максимум 10 баллов)

Реализуйте обучение и тестирование нейронной сети для предоставленного вам набора данных. Соотношение между полученным значением метрики на тестовой выборке и баллами за задание следующее:

- $\text{RMSE} \le 9.00 $ &mdash; 4 балла
- $\text{RMSE} \le 8.90 $ &mdash; 6 баллов
- $\text{RMSE} \le 8.80 $ &mdash; 8 баллов
- $\text{RMSE} \le 8.75 $ &mdash; 10 баллов

Есть несколько правил, которых вам нужно придерживаться:

- Весь пайплайн обучения должен быть написан на PyTorch. При этом вы можете пользоваться другими библиотеками (`numpy`, `sklearn` и пр.), но только для обработки данных. То есть как угодно трансформировать данные и считать метрики с помощью этих библиотек можно, а импортировать модели из `sklearn` и выбивать с их помощью требуемое качество &mdash; нельзя. Также нельзя пользоваться библиотеками, для которых сам PyTorch является зависимостью.

- Мы никак не ограничиваем ваш выбор архитектуры модели, но скорее всего вам будет достаточно полносвязной нейронной сети.

- Для обучения запрещается использовать какие-либо иные данные, кроме обучающей выборки.

- Ансамблирование моделей запрещено.

### Полезные советы:

- Очень вряд ли, что у вас с первого раза получится выбить качество на 10 баллов, поэтому пробуйте разные архитектуры, оптимизаторы и значения гиперпараметров. В идеале при запуске каждого нового эксперимента вы должны менять что-то одно, чтобы точно знать, как этот фактор влияет на качество.

- Не забудьте, что для улучшения качества модели вам поможет **нормировка таргета**.

- Тот факт, что мы занимаемся глубинным обучением, не означает, что стоит забывать про приемы, использующиеся в классическом машинном обучении. Так что обязательно проводите исследовательский анализ данных, отрисовывайте нужные графики и не забывайте про масштабирование и подбор гиперпараметров.

- Вы наверняка столкнетесь с тем, что ваша нейронная сеть будет сильно переобучаться. Для нейросетей существуют специальные методы регуляризации, например, dropout ([статья](https://jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf)) и weight decay ([блогпост](https://towardsdatascience.com/weight-decay-l2-regularization-90a9e17713cd)). Они, разумеется, реализованы в PyTorch. Попробуйте поэкспериментировать с ними.

- Если вы чего-то не знаете, не гнушайтесь гуглить. В интернете очень много полезной информации, туториалов и советов по глубинному обучению в целом и по PyTorch в частности. Но не забывайте, что за скатанный код без ссылки на источник придется ответить по всей строгости!

- Если вы сразу реализуете обучение на GPU, то у вас будет больше времени на эксперименты, так как любые вычисления будут работать быстрее. Google Colab предоставляет несколько GPU-часов (обычно около 8-10) в сутки бесплатно.

- Чтобы отладить код, можете обучаться на небольшой части данных или даже на одном батче. Если лосс на обучающей выборке не падает, то что-то точно идет не так!

- Пользуйтесь утилитами, которые вам предоставляет PyTorch (например, Dataset и Dataloader). Их специально разработали для упрощения разработки пайплайна обучения.

- Скорее всего вы захотите отслеживать прогресс обучения. Для создания прогресс-баров есть удобная библиотека `tqdm`.

- Быть может, вы захотите, чтобы графики рисовались прямо во время обучения. Можете воспользоваться функцией [clear_output](http://ipython.org/ipython-doc/dev/api/generated/IPython.display.html#IPython.display.clear_output), чтобы удалять старый график и рисовать новый на его месте.

**ОБЯЗАТЕЛЬНО** рисуйте графики зависимости лосса/метрики на обучающей и тестовой выборках в зависимости от времени обучения. Если обучение занимает относительно небольшое число эпох, то лучше рисовать зависимость от номера шага обучения, если же эпох больше, то рисуйте зависимость по эпохам. Если проверяющий не увидит такого графика для вашей лучшей модели, то он в праве снизить баллы за задание.

**ВАЖНО!** Ваше решение должно быть воспроизводимым. Если это не так, то проверяющий имеет право снизить баллы за задание. Чтобы зафиксировать random seed, воспользуйтесь функцией из ячейки ниже.



In [7]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

Вы можете придерживаться любой адекватной струкуры кода, но мы советуем воспользоваться следующими сигнатурами функций. Лучше всего, если вы проверите ваши предсказания ассертом: так вы убережете себя от разных косяков, например, что вектор предсказаний состоит из всего одного числа. В любом случае, внимательно следите за тем, для каких тензоров вы считаете метрику RMSE. При случайном или намеренном введении в заблуждение проверяющие очень сильно разозлятся.

In [8]:
import matplotlib.pyplot as plt

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515345 entries, 0 to 515344
Data columns (total 91 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   0       515345 non-null  int64  
 1   1       515345 non-null  float64
 2   2       515345 non-null  float64
 3   3       515345 non-null  float64
 4   4       515345 non-null  float64
 5   5       515345 non-null  float64
 6   6       515345 non-null  float64
 7   7       515345 non-null  float64
 8   8       515345 non-null  float64
 9   9       515345 non-null  float64
 10  10      515345 non-null  float64
 11  11      515345 non-null  float64
 12  12      515345 non-null  float64
 13  13      515345 non-null  float64
 14  14      515345 non-null  float64
 15  15      515345 non-null  float64
 16  16      515345 non-null  float64
 17  17      515345 non-null  float64
 18  18      515345 non-null  float64
 19  19      515345 non-null  float64
 20  20      515345 non-null  float64
 21  21      51

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

In [11]:
X = df_scaled.iloc[:, 1:].values
y = df_scaled.iloc[:, 0].values

train_size = 463715
X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

In [12]:
model.fit (X_train, y_train)

y_pred = model.predict (X_test)

rmse = mean_squared_error (y_test, y_pred) ** 0.5
print ('rmse: ', rmse)

rmse:  0.10685578080570851


In [13]:
y_orig = df.iloc[:, 0].values
y_max = y_orig.max()
y_min = y_orig.min()

rmse_original = rmse * (y_max - y_min)
print (rmse_original)

9.510164491708057


In [14]:
from sklearn.decomposition import PCA

In [15]:
pca = PCA(.90)
pca.fit(X_train)
pca.n_components_

np.int64(39)

In [16]:
X_train1 = pca.transform (X_train)
X_test1 = pca.transform (X_test)

y_train1 = y_train - min (y_train)
y_test1 = y_test - min (y_test)

In [17]:
model.fit (X_train1, y_train1)

y_pred = model.predict (X_test1)

rmse = mean_squared_error (y_test1, y_pred) ** 0.5
rmse_original = rmse * (y_max - y_min)
print ('rmse: ', rmse_original)

rmse:  10.821064686057067


In [18]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

X_group1 = X[:, :12]  #первые 12
X_group2 = X[:, 12:]  #последние 78

pca1 = PCA(n_components=6)
X_pca1 = pca1.fit_transform(X_group1)
pca2 = PCA(n_components=39)
X_pca2 = pca2.fit_transform(X_group2)

X_combined = np.hstack([X_pca1, X_pca2])

X_train = X_combined[:train_size, :]
y_train = y[:train_size]
X_test = X_combined[train_size:, :]
y_test = y[train_size:]

In [19]:
model.fit (X_train, y_train)

y_pred = model.predict (X_test)

rmse = mean_squared_error (y_test, y_pred) ** 0.5
print ('rmse: ', rmse)

rmse:  10.12629216036728


In [20]:
import torch
import torch.nn.functional as F
from torch import nn
import torch.optim as optim
import torch.utils.data as data_utils

In [38]:
df = pd.read_csv("data.txt.zip", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,81,82,83,84,85,86,87,88,89,90
0,2001,49.94357,21.47114,73.07750,8.74861,-17.40628,-13.09905,-25.01202,-12.23257,7.83089,...,13.01620,-54.40548,58.99367,15.37344,1.11144,-23.08793,68.40795,-1.82223,-27.46348,2.26327
1,2001,48.73215,18.42930,70.32679,12.94636,-10.32437,-24.83777,8.76630,-0.92019,18.76548,...,5.66812,-19.68073,33.04964,42.87836,-9.90378,-32.22788,70.49388,12.04941,58.43453,26.92061
2,2001,50.95714,31.85602,55.81851,13.41693,-6.57898,-18.54940,-3.27872,-2.35035,16.07017,...,3.03800,26.05866,-50.92779,10.93792,-0.07568,43.20130,-115.00698,-0.05859,39.67068,-0.66345
3,2001,48.24750,-1.89837,36.29772,2.58776,0.97170,-26.21683,5.05097,-10.34124,3.55005,...,34.57337,-171.70734,-16.96705,-46.67617,-12.51516,82.58061,-72.08993,9.90558,199.62971,18.85382
4,2001,50.97020,42.20998,67.09964,8.46791,-15.85279,-16.81409,-12.48207,-9.37636,12.63699,...,9.92661,-55.95724,64.92712,-17.72522,-1.49237,-7.50035,51.76631,7.88713,55.66926,28.74903


In [39]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

train_size = 463715
X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

In [23]:
scaler = MinMaxScaler()
y = scaler.fit_transform (y.reshape(-1,1))

print (y)

[[0.88764045]
 [0.88764045]
 [0.88764045]
 ...
 [0.94382022]
 [0.94382022]
 [0.93258427]]


In [24]:

X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

In [25]:
model.fit (X_train, y_train)

y_pred = model.predict (X_test)

rmse = mean_squared_error (y_test, y_pred) ** 0.5
rmse_original = rmse * (y_max - y_min)
print ('rmse: ', rmse_original)

rmse:  9.510160711373398


In [26]:
class NeuralNet(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid (self.fc3 (x))
        return x.squeeze(-1)

model = NeuralNet(input_dim=90)

In [27]:
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [28]:
input_size = 90
learning_rate = 1e-6
epochs = 100

In [29]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [30]:
model = NeuralNet (input_size)

In [31]:
from torch.utils.data import TensorDataset, DataLoader
batch_size = 32

y_tensor = y_tensor.squeeze (-1)
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [34]:
for epoch in range (epochs):

    model.train()
    epoch_loss = 0.0

    for inputs, labels in train_loader:

        inputs, labels = inputs, labels
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        predictions = (outputs > 0.5).float()
        epoch_loss += loss.item()

    if epoch % 10 == 9:
      print(f"Epoch {epoch}: Loss = {epoch_loss:.4f}")


ValueError: Using a target size (torch.Size([32, 1])) that is different to the input size (torch.Size([32])) is deprecated. Please ensure they have the same size.

In [ ]:
print (epoch_loss)

125025.36375808716


In [35]:
df = pd.read_csv("data.txt.zip", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,81,82,83,84,85,86,87,88,89,90
0,2001,49.94357,21.47114,73.07750,8.74861,-17.40628,-13.09905,-25.01202,-12.23257,7.83089,...,13.01620,-54.40548,58.99367,15.37344,1.11144,-23.08793,68.40795,-1.82223,-27.46348,2.26327
1,2001,48.73215,18.42930,70.32679,12.94636,-10.32437,-24.83777,8.76630,-0.92019,18.76548,...,5.66812,-19.68073,33.04964,42.87836,-9.90378,-32.22788,70.49388,12.04941,58.43453,26.92061
2,2001,50.95714,31.85602,55.81851,13.41693,-6.57898,-18.54940,-3.27872,-2.35035,16.07017,...,3.03800,26.05866,-50.92779,10.93792,-0.07568,43.20130,-115.00698,-0.05859,39.67068,-0.66345
3,2001,48.24750,-1.89837,36.29772,2.58776,0.97170,-26.21683,5.05097,-10.34124,3.55005,...,34.57337,-171.70734,-16.96705,-46.67617,-12.51516,82.58061,-72.08993,9.90558,199.62971,18.85382
4,2001,50.97020,42.20998,67.09964,8.46791,-15.85279,-16.81409,-12.48207,-9.37636,12.63699,...,9.92661,-55.95724,64.92712,-17.72522,-1.49237,-7.50035,51.76631,7.88713,55.66926,28.74903


In [ ]:
for i in range (0, 50):
  df = df[df.index % 3 != 0]

# Где-то здесь я начал всё заново

In [36]:
learning_rate = 0.01
epochs = 100

class LeModel (nn.Module):
    def __init__(self, input_dim=90, output_dim=1):
        super(LeModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [37]:
model = LeModel()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()


In [42]:
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [46]:
for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs, targets = batch
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/100], Loss: 36364.4813
Epoch [2/100], Loss: 30200.5184
Epoch [3/100], Loss: 18716.6363
Epoch [4/100], Loss: 15480.4385
Epoch [5/100], Loss: 4267.6919
Epoch [6/100], Loss: 184.1574
Epoch [7/100], Loss: 183.0850
Epoch [8/100], Loss: 346.6206
Epoch [9/100], Loss: 136.2440
Epoch [10/100], Loss: 135.7153
Epoch [11/100], Loss: 127.6765
Epoch [12/100], Loss: 127.9764
Epoch [13/100], Loss: 128.1271
Epoch [14/100], Loss: 127.3327
Epoch [15/100], Loss: 127.6111
Epoch [16/100], Loss: 127.5097
Epoch [17/100], Loss: 126.9176
Epoch [18/100], Loss: 127.3815


KeyboardInterrupt: 

Лосс приемлемый, надо смотреть на более маленькой выборке (чтобы быстрее обучилось)

In [47]:
df = pd.read_csv("data.txt.zip", header=None)
shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [49]:
df_small = shuffled_df.iloc[:100000].copy()

In [50]:
X = df_small.iloc[:, 1:].values
y = df_small.iloc[:, 0].values

train_size = 70000
X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

In [51]:
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [52]:
epochs = 30

for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs, targets = batch
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/30], Loss: 125.8664
Epoch [2/30], Loss: 125.9660
Epoch [3/30], Loss: 125.3257
Epoch [4/30], Loss: 126.3994
Epoch [5/30], Loss: 125.1566
Epoch [6/30], Loss: 125.0035
Epoch [7/30], Loss: 126.5909
Epoch [8/30], Loss: 125.6995
Epoch [9/30], Loss: 126.0406
Epoch [10/30], Loss: 125.9312
Epoch [11/30], Loss: 125.4894
Epoch [12/30], Loss: 125.4110
Epoch [13/30], Loss: 125.6653
Epoch [14/30], Loss: 125.4747
Epoch [15/30], Loss: 125.2926
Epoch [16/30], Loss: 125.6645
Epoch [17/30], Loss: 124.6869
Epoch [18/30], Loss: 124.4001
Epoch [19/30], Loss: 125.5101
Epoch [20/30], Loss: 125.6432
Epoch [21/30], Loss: 125.7670
Epoch [22/30], Loss: 126.4456
Epoch [23/30], Loss: 125.3049
Epoch [24/30], Loss: 124.6084
Epoch [25/30], Loss: 124.9751
Epoch [26/30], Loss: 125.0508
Epoch [27/30], Loss: 126.1031
Epoch [28/30], Loss: 126.7854
Epoch [29/30], Loss: 124.3049
Epoch [30/30], Loss: 125.1597


In [55]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).squeeze()
    rmse = torch.sqrt(torch.mean((preds - y_test_tensor) ** 2))
    print(f"Test RMSE: {rmse.item():.4f}")

Test RMSE: 11.0662


Не слишком плохо для обрезанного датасета и 30 эпох. Может, это даже хорошая модель?

In [56]:
#Поработаем немного с датасетом

from sklearn.preprocessing import StandardScaler

X = df_small.iloc[:, 1:].values
y = df_small.iloc[:, 0].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [57]:
train_size = 70000
X_train = X_scaled[:train_size, :]
y_train = y[:train_size]
X_test = X_scaled[train_size:, :]
y_test = y[train_size:]

In [61]:
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True) #увеличил batch_size, увеличу и эпохи в этот раз

In [63]:
learning_rate = 0.001 #попробуем такой, возможно, будет лучше
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [64]:
epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs, targets = batch
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/50], Loss: 49138.3894
Epoch [2/50], Loss: 1964.2057
Epoch [3/50], Loss: 900.1444
Epoch [4/50], Loss: 533.7682
Epoch [5/50], Loss: 359.9019
Epoch [6/50], Loss: 259.2619
Epoch [7/50], Loss: 199.5226
Epoch [8/50], Loss: 164.4712
Epoch [9/50], Loss: 141.9803
Epoch [10/50], Loss: 131.3611
Epoch [11/50], Loss: 121.8406
Epoch [12/50], Loss: 115.4391
Epoch [13/50], Loss: 110.8660
Epoch [14/50], Loss: 107.8972
Epoch [15/50], Loss: 104.5063
Epoch [16/50], Loss: 102.0186
Epoch [17/50], Loss: 99.6543
Epoch [18/50], Loss: 98.1172
Epoch [19/50], Loss: 97.4673
Epoch [20/50], Loss: 96.2852
Epoch [21/50], Loss: 94.5160
Epoch [22/50], Loss: 94.2621
Epoch [23/50], Loss: 93.8085
Epoch [24/50], Loss: 93.9705
Epoch [25/50], Loss: 92.3019
Epoch [26/50], Loss: 92.3256
Epoch [27/50], Loss: 92.4113
Epoch [28/50], Loss: 91.8406
Epoch [29/50], Loss: 89.3742
Epoch [30/50], Loss: 90.2492
Epoch [31/50], Loss: 89.5791
Epoch [32/50], Loss: 88.6612
Epoch [33/50], Loss: 88.8788
Epoch [34/50], Loss: 88.0256
Epoc

In [66]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).squeeze()
    rmse = torch.sqrt(torch.mean((preds - y_test_tensor) ** 2))
    print(f"Test RMSE: {rmse.item():.4f}")

Test RMSE: 12.2833


In [67]:
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs, targets = batch
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/50], Loss: 88.3157
Epoch [2/50], Loss: 87.4367
Epoch [3/50], Loss: 87.1046
Epoch [4/50], Loss: 86.6162
Epoch [5/50], Loss: 86.3105
Epoch [6/50], Loss: 86.3365
Epoch [7/50], Loss: 86.0817
Epoch [8/50], Loss: 86.2141
Epoch [9/50], Loss: 85.7899
Epoch [10/50], Loss: 85.4726
Epoch [11/50], Loss: 85.0387
Epoch [12/50], Loss: 84.4142
Epoch [13/50], Loss: 84.6482
Epoch [14/50], Loss: 84.3404
Epoch [15/50], Loss: 84.1694
Epoch [16/50], Loss: 83.4924
Epoch [17/50], Loss: 83.4691
Epoch [18/50], Loss: 83.6871
Epoch [19/50], Loss: 83.0438
Epoch [20/50], Loss: 83.1438
Epoch [21/50], Loss: 82.8887
Epoch [22/50], Loss: 82.9456
Epoch [23/50], Loss: 82.5529
Epoch [24/50], Loss: 81.6309
Epoch [25/50], Loss: 82.2673
Epoch [26/50], Loss: 81.3619
Epoch [27/50], Loss: 82.0549
Epoch [28/50], Loss: 81.5330
Epoch [29/50], Loss: 81.2133
Epoch [30/50], Loss: 81.5766
Epoch [31/50], Loss: 80.6357
Epoch [32/50], Loss: 80.8976
Epoch [33/50], Loss: 81.1333
Epoch [34/50], Loss: 80.7599
Epoch [35/50], Loss: 80

In [68]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).squeeze()
    rmse = torch.sqrt(torch.mean((preds - y_test_tensor) ** 2))
    print(f"Test RMSE: {rmse.item():.4f}")

Test RMSE: 9.1443


9.14 для урезанного датасета??! Можно попробовать на полноразмерном.

In [69]:
X = shuffled_df.iloc[:, 1:].values
y = shuffled_df.iloc[:, 0].values
X_scaled = scaler.fit_transform(X)

train_size = 463715
X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

In [70]:
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs, targets = batch
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/50], Loss: 13499.3459
Epoch [2/50], Loss: 1378.8941
Epoch [3/50], Loss: 767.1681
Epoch [4/50], Loss: 262.9335
Epoch [5/50], Loss: 231.8769
Epoch [6/50], Loss: 179.0150
Epoch [7/50], Loss: 133.4431
Epoch [8/50], Loss: 131.5065
Epoch [9/50], Loss: 133.6850
Epoch [10/50], Loss: 142.3942
Epoch [11/50], Loss: 122.6061
Epoch [12/50], Loss: 130.5429
Epoch [13/50], Loss: 131.1972
Epoch [14/50], Loss: 127.7308
Epoch [15/50], Loss: 142.9112
Epoch [16/50], Loss: 120.4943
Epoch [17/50], Loss: 124.3704
Epoch [18/50], Loss: 124.7726
Epoch [19/50], Loss: 120.7573
Epoch [20/50], Loss: 122.8419
Epoch [21/50], Loss: 125.0720
Epoch [22/50], Loss: 126.6289
Epoch [23/50], Loss: 121.9673
Epoch [24/50], Loss: 120.7850
Epoch [25/50], Loss: 131.0618
Epoch [26/50], Loss: 121.6974
Epoch [27/50], Loss: 121.2491
Epoch [28/50], Loss: 123.1940
Epoch [29/50], Loss: 124.1328
Epoch [30/50], Loss: 120.7611
Epoch [31/50], Loss: 120.4540
Epoch [32/50], Loss: 121.6184
Epoch [33/50], Loss: 120.6526
Epoch [34/50], L

In [73]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).squeeze()
    rmse = torch.sqrt(torch.mean((preds - y_test_tensor) ** 2))
    print(f"Test RMSE: {rmse.item():.4f}")

Test RMSE: 11.0271


Получилось не очень, возможно, размер batch подвёл.

## Задание 2. (0 баллов, но при невыполнении максимум за все задание &mdash; 0 баллов)

Напишите небольшой отчет о том, как вы добились полученного качества: какие средства использовали и какие эксперименты проводили. Подробно расскажите об архитектурах и значениях гиперпараметров, а также какие метрики на тесте они показывали. Чтобы отчет был зачтен, необходимо привести хотя бы 3 эксперимента.

In [ ]:
Бейслайн: 9.5/ 10.1

Первая модель: слишком огромный loss, даже не закончил обучение. Полносвязная, 3 слоя (90 -> 64 -> ReLU -> 32 -> ReLU -> Sigmoid). Не получилась.

Вторая модель: 90 -> 64 -> ReLU -> Dropout -> 32 -> ReLU -> Output
Лучший результат: 9.1443 на урезанном датасете
LR = 0.001, эпохи = 50 (потому что 100 очень долго ждать =( )
